Note: I removed my API key from this document so the notebook will not run without an API key being defined.

RQ1 - How many wine shops are in Denver?

- Using the Google Maps API to collect a starting point of wine retailers in Denver, CO
- Focusing on wine retailers, specifically. Not bars/restaurants that serve wine - as the premise is "wine that I can buy a bottle of (and take home)"

Google Maps API Docs: https://developers.google.com/maps/documentation/places/web-service

In [1]:
%pip install --quiet python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [ ]:
#setup 
import pandas as pd
import requests
import numpy as np
import os
from dotenv import load_dotenv

api_key = os.getenv('google_api_key')
my_address = os.getenv('my_address')

Finding the liquor stores in Denver:

- Denver Metro Area "center" = 39.742043 lat, -104.991531 lon per https://www.latlong.net/place/denver-co-usa-3174.html
- Denver spans ~154 sq miles per https://web.archive.org/web/20230305171359/http://www.usa.com/denver-co.htm

In [ ]:
#Lat/lon of my address
# This cell is intentionally redacted and will not print a home address.
my_coords = [None, None]
my_coords

[39.728886, -104.9715073]

In [4]:
df_denver_liquor = pd.read_csv("denver_liquor-licenses.csv")

In [5]:
df_denver_liquor.columns

Index(['OBJECTID', 'BFN', 'BUS_PROF_NAME', 'FULL_ADDRESS', 'LICENSES',
       'LIC_STATUS', 'ISSUE_DATE', 'END_DATE', 'ADDRESS_ID', 'ADDRESS_LINE1',
       'ADDRESS_LINE2', 'CITY', 'STATE', 'ZIP', 'COUNCIL_DIST', 'POLICE_DIST',
       'CENSUS_TRACT', 'X_COORD', 'Y_COORD', 'GLOBALID', 'NEIGHBORHOOD',
       'ZONE_DISTRICT', 'HEARING_DATE', 'HEARING_TIME', 'HEARING_STATUS',
       'HEARING_TIMESTAMP', 'x', 'y'],
      dtype='str')

In [6]:
df_denver_liquor.shape

(10076, 28)

In [7]:
df_denver_liquor_cleaned = df_denver_liquor.dropna(axis=1, how='all')

In [8]:
df_denver_liquor_cleaned.columns

Index(['OBJECTID', 'BFN', 'BUS_PROF_NAME', 'FULL_ADDRESS', 'LICENSES',
       'LIC_STATUS', 'ISSUE_DATE', 'END_DATE', 'ADDRESS_ID', 'ADDRESS_LINE1',
       'CITY', 'STATE', 'ZIP', 'COUNCIL_DIST', 'POLICE_DIST', 'CENSUS_TRACT',
       'X_COORD', 'Y_COORD', 'GLOBALID', 'NEIGHBORHOOD', 'ZONE_DISTRICT',
       'HEARING_DATE', 'HEARING_TIME', 'HEARING_STATUS', 'HEARING_TIMESTAMP',
       'x', 'y'],
      dtype='str')

In [9]:
df_denver_liquor_cleaned['LICENSES'].value_counts()

LICENSES
LIQUOR - SPECIAL EVENTS                                             5759
LIQUOR - HOTEL AND RESTAURANT                                       1258
LIQUOR - TASTINGS                                                    461
LIQUOR - HOTEL AND RESTAURANT AND CABARET                            386
LIQUOR - STORE                                                       306
LIQUOR - TAVERN AND CABARET                                          288
LIQUOR - TAVERN                                                      286
LIQUOR - BEER & WINE                                                 268
LIQUOR - FERMENTED MALT BEVERAGE                                     257
LIQUOR - ART GALLERY PERMIT                                          159
LIQUOR - HOTEL/RESTAURANT                                            156
LIQUOR - 3.2% BEER                                                    57
LIQUOR - 3.2 % BEER                                                   51
LIQUOR - RETAIL                           

# Type of Liquor Licenses:
source: https://content.leg.colorado.gov/sites/default/files/2022_update_liquor_handbook_with_cover.pdf 

### Off-premises retail licenses and permits.
These licensed retailers may sell alcohol in original sealed containers for off-premises consumption.

### On-premises retail licenses and permits.
Establishments that sell alcohol for consumption on the licensed premises.

### Hotel and restaurant license.
These licenses are issued to hotels and restaurants selling alcohol beverages to customers.

### Lodging and entertainment facility license.
These licenses are issued to either a lodging facility, wherethe primary business is to provide the public with sleeping rooms and meeting facilities, or an
entertainment facility, where the primary business is to provide the public with sports or entertainment activities. 

### Optional premises license.
These licenses are issued to an outdoor sports and recreational facility

### Tastings licenses.
An alcohol beverage tastings permit allows a retail liquor store or liquor-licensed drugstore to conduct sample tastings of alcohol within their establishment. Tastings are not to exceed a total of 5 hours per day and they can only be held between the hours of 11 a.m. to 9 p.m.

https://www.denvergov.org/Government/Agencies-Departments-Offices/Agencies-Departments-Offices-Directory/Business-Licensing/Business-licenses/Liquor/Tastings-permit

## Methodology notes on license filtering

For the purposes of this project, I am filtering for licenses that are 'LIQUOR - STORE' or 'LIQUOR - TASTINGS'. 'LIQUOR - STORE' license types indicate they are able to sell alcohol for off-premises consumption. In the case a retail liquor store has a 'LIQUOR - TASTINGS' license, they are represented as license type, 'LIQUOR- TASTINGS' in this data table.

In [10]:
df_denver_liquor_stores = df_denver_liquor_cleaned[df_denver_liquor_cleaned['LICENSES'].isin(['LIQUOR - STORE', 'LIQUOR - TASTINGS'])]

In [11]:
df_denver_liquor_stores

,OBJECTID,BFN,BUS_PROF_NAME,FULL_ADDRESS,LICENSES,LIC_STATUS,ISSUE_DATE,END_DATE,ADDRESS_ID,ADDRESS_LINE1,...,Y_COORD,GLOBALID,NEIGHBORHOOD,ZONE_DISTRICT,HEARING_DATE,HEARING_TIME,HEARING_STATUS,HEARING_TIMESTAMP,x,y
2353,10007896,2022-BFN-0022631,COST PLUS WORLD MARKET,8601 W CROSS DR UNIT CP,LIQUOR - STORE,LICENSE ISSUED - ACTIVE,2/18/2026 9:47:22 AM,4/21/2027 12:00:00 AM,32733059.0,8601 W Cross Dr Unit CP,...,1649458.0,eddd4d1f-99c0-46b2-b072-4f2af3d9a76d,Marston,B-4,NaN,NaN,NaN,NaN,3.114335e+06,1.649458e+06
2355,10007898,2017-BFN-0008313,SOUTHWEST LIQUOR,8500 W CRESTLINE AVE UNIT G2,LIQUOR - STORE,CLOSED - ADMINISTRATIVELY,1/26/2018 9:52:11 AM,10/31/2017 3:29:40 PM,264321.0,8500 W Crestline Ave Unit G2,...,1649679.0,8fba456c-6d05-414e-8d8b-7df6cb16f57e,Marston,B-4,NaN,NaN,NaN,NaN,3.114751e+06,1.649679e+06
2357,10007900,2008-BFN-1032542,SOUTHWEST WINE AND LIQUORS,8500 W CRESTLINE AVE # G-2,LIQUOR - STORE,CLOSED - EXPIRED,7/21/2017 12:00:00 AM,7/21/2017 12:00:00 AM,351830.0,8500 W Crestline Ave # G-2,...,1650075.0,893f76c3-edd1-4cd6-9b91-dfaded453237,Marston,B-4,NaN,NaN,NaN,NaN,3.114607e+06,1.650075e+06
2358,10007901,2006-BFN-1021689,L&C LIQUORS,8100 W CRESTLINE AVE UNIT A-18,LIQUOR - STORE,LICENSE ISSUED - ACTIVE,10/31/2025 10:58:22 AM,12/14/2026 12:00:00 AM,264317.0,8100 W Crestline Ave Unit A18,...,1650209.0,8a09d97d-fbed-4d26-a721-a4200f27e2bc,Marston,B-4,NaN,NaN,NaN,NaN,3.115317e+06,1.650209e+06
2365,10007908,2021-BFN-0000048,CHE POCHO INC,8555 W BELLEVIEW AVE STE A03,LIQUOR - TASTINGS,CLOSED - EXPIRED,4/8/2022 7:28:00 AM,1/7/2022 12:00:00 AM,270011.0,8555 W Belleview Ave Ste A03,...,1652660.0,c2ec0119-5e61-42db-8a12-c97184d3e988,Marston,B-2,NaN,NaN,NaN,NaN,3.114424e+06,1.652660e+06
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9944,10015487,2025-BFN-0027819,MOONLIGHT LIQUORS,6407 N TOWER 120,LIQUOR - STORE,PENDING,12/23/2025 12:00:00 AM,12/17/2025 2:38:50 PM,32789598.0,6407 N Tower Rd Ste 120,...,1721884.0,d079f51e-ff4b-4296-8806-7538282c9f24,DIA,C-MU-10,NaN,NaN,NaN,NaN,3.204321e+06,1.721884e+06
9989,10015532,2011-BFN-1056745,ARRON'S LIQUORS,7111 N TOWER RD,LIQUOR - STORE,CLOSED - SURRENDERED,8/26/2020 10:06:18 AM,2/14/2021 12:00:00 AM,376285.0,7111 N Tower Rd,...,1726579.0,376686c2-46d7-455b-ab74-e6c1a2858baa,DIA,S-MX-8,NaN,NaN,NaN,NaN,3.204256e+06,1.726579e+06
10006,10015549,2012-BFN-1062890,WINE,8400 PENA BLVD CC A,LIQUOR - STORE,CLOSED - EXPIRED,7/2/2018 3:00:03 AM,4/2/2018 12:00:00 AM,221153.0,8400 Pena Blvd,...,1735292.0,0d7b1306-70c4-46a8-8bca-39e1d5af6163,DIA,DIA,NaN,NaN,NaN,NaN,3.231977e+06,1.735292e+06
10051,10015594,2025-BFN-0015684,MILE HIGH GAME DAY,8900 PENA BLVD # BC-5,LIQUOR - STORE,PENDING,3/23/2026 12:00:00 AM,7/17/2025 10:11:09 AM,263709.0,8900 Pena Blvd Bldg BC # 5,...,1738784.0,8771b996-507d-4cde-a799-60c765652d82,DIA,DIA,NaN,NaN,NaN,NaN,3.231370e+06,1.738784e+06


In [12]:
import regex
df_denver_liquor_stores_active = df_denver_liquor_stores[df_denver_liquor_stores['LIC_STATUS'].str.contains('ACTIVE', case=False)]

In [13]:
df_denver_liquor_stores_active['LIC_STATUS'].unique()

<StringArray>
['LICENSE ISSUED - ACTIVE']
Length: 1, dtype: str

In [14]:
len(df_denver_liquor_stores_active)

221

In [15]:
df_denver_liquor_stores_active.head()

,OBJECTID,BFN,BUS_PROF_NAME,FULL_ADDRESS,LICENSES,LIC_STATUS,ISSUE_DATE,END_DATE,ADDRESS_ID,ADDRESS_LINE1,...,Y_COORD,GLOBALID,NEIGHBORHOOD,ZONE_DISTRICT,HEARING_DATE,HEARING_TIME,HEARING_STATUS,HEARING_TIMESTAMP,x,y
2353,10007896,2022-BFN-0022631,COST PLUS WORLD MARKET,8601 W CROSS DR UNIT CP,LIQUOR - STORE,LICENSE ISSUED - ACTIVE,2/18/2026 9:47:22 AM,4/21/2027 12:00:00 AM,32733059.0,8601 W Cross Dr Unit CP,...,1649458.0,eddd4d1f-99c0-46b2-b072-4f2af3d9a76d,Marston,B-4,NaN,NaN,NaN,NaN,3.114335e+06,1.649458e+06
2358,10007901,2006-BFN-1021689,L&C LIQUORS,8100 W CRESTLINE AVE UNIT A-18,LIQUOR - STORE,LICENSE ISSUED - ACTIVE,10/31/2025 10:58:22 AM,12/14/2026 12:00:00 AM,264317.0,8100 W Crestline Ave Unit A18,...,1650209.0,8a09d97d-fbed-4d26-a721-a4200f27e2bc,Marston,B-4,NaN,NaN,NaN,NaN,3.115317e+06,1.650209e+06
2370,10007913,2025-BFN-0011908,MARTINI MARTINE LLC,8555 W BELLEVIEW AVE STE A03,LIQUOR - TASTINGS,LICENSE ISSUED - ACTIVE,6/16/2025 12:00:00 AM,6/16/2026 12:00:00 AM,270011.0,8555 W Belleview Ave Ste A03,...,1652660.0,5911c82c-d94b-450f-bdd7-9a39dd7fe1c1,Marston,B-2,NaN,NaN,NaN,NaN,3.114424e+06,1.652660e+06
2380,10007923,2011-BFN-1057097,BELLEVIEW WINE AND SPIRITS,7795 E BELLEVIEW 3,LIQUOR - STORE,LICENSE ISSUED - ACTIVE,2/24/2026 8:36:29 AM,2/12/2027 12:00:00 AM,32690148.0,7795 E Belleview Ave Unit 3,...,1652956.0,6710414d-cbc7-4a3a-9371-91cb04db16ef,Hampden South,B-8,NaN,NaN,NaN,NaN,3.169520e+06,1.652956e+06
2386,10007929,2011-BFN-1055803,VILLAGE WEST LIQUOR,8555 W BELLEVIEW AVE SUITE A03,LIQUOR - STORE,LICENSE ISSUED - ACTIVE,6/3/2025 10:24:42 AM,5/20/2026 12:00:00 AM,155994.0,8555 W Belleview Ave,...,1653012.0,59297ab8-88d5-4193-ac3d-078aeed9e791,Marston,B-2,NaN,NaN,NaN,NaN,3.114526e+06,1.653012e+06


In [24]:
df_denver_liquor_stores_active_clean.to_csv('denver_liquor_stores.csv')

In [17]:
df_denver_liquor_stores_active.columns

Index(['OBJECTID', 'BFN', 'BUS_PROF_NAME', 'FULL_ADDRESS', 'LICENSES',
       'LIC_STATUS', 'ISSUE_DATE', 'END_DATE', 'ADDRESS_ID', 'ADDRESS_LINE1',
       'CITY', 'STATE', 'ZIP', 'COUNCIL_DIST', 'POLICE_DIST', 'CENSUS_TRACT',
       'X_COORD', 'Y_COORD', 'GLOBALID', 'NEIGHBORHOOD', 'ZONE_DISTRICT',
       'HEARING_DATE', 'HEARING_TIME', 'HEARING_STATUS', 'HEARING_TIMESTAMP',
       'x', 'y'],
      dtype='str')

In [18]:
df_denver_liquor_stores_active_clean = df_denver_liquor_stores_active.drop(columns = [
    'OBJECTID', 'BFN', 'HEARING_DATE', 'HEARING_TIME', 'HEARING_STATUS', 'HEARING_TIMESTAMP'
]
    
)

## Liquor store location data

This dataset contains several markers for location data, including:
- Full address
- A "Global ID"
- 'x' - x coordinates
- 'y' - y coordinates

### Colorado government use of 'x' and 'y' coordinates

Colorado government data is provided using Colorado State Plane Coordinates. To get lat/lon, I can either convert the 'x' and 'y' accordingly, or use the address geolocator.


In [ ]:
#Lat/lon of my address
# This cell is intentionally redacted and will not print a home address.
my_coords = [None, None]
my_coords

[39.728886, -104.9715073]

pyproj to transform Colorado State Plane to lat/lon: https://pyproj4.github.io/pyproj/stable/api/transformer.html#pyproj.transformer.Transformer.transform

In [20]:
%pip install pyproj

Note: you may need to restart the kernel to use updated packages.


In [21]:
from pyproj import Transformer

transformer = Transformer.from_crs("EPSG:2232", "EPSG:4326", always_xy=True)

df_denver_liquor_stores_active_clean['lon'], df_denver_liquor_stores_active_clean['lat'] = transformer.transform(
    df_denver_liquor_stores_active_clean['x'].values,
    df_denver_liquor_stores_active_clean['y'].values
)

print(df_denver_liquor_stores_active_clean[['x', 'y', 'lon', 'lat']].head(3))

                 x             y         lon        lat
2353  3.114335e+06  1.649458e+06 -105.094152  39.615852
2358  3.115317e+06  1.650209e+06 -105.090655  39.617902
2370  3.114424e+06  1.652660e+06 -105.093786  39.624642


In [22]:
df_denver_liquor_stores_active_clean.head()

,BUS_PROF_NAME,FULL_ADDRESS,LICENSES,LIC_STATUS,ISSUE_DATE,END_DATE,ADDRESS_ID,ADDRESS_LINE1,CITY,STATE,...,CENSUS_TRACT,X_COORD,Y_COORD,GLOBALID,NEIGHBORHOOD,ZONE_DISTRICT,x,y,lon,lat
2353,COST PLUS WORLD MARKET,8601 W CROSS DR UNIT CP,LIQUOR - STORE,LICENSE ISSUED - ACTIVE,2/18/2026 9:47:22 AM,4/21/2027 12:00:00 AM,32733059.0,8601 W Cross Dr Unit CP,Denver,CO,...,12010.0,3114335.0,1649458.0,eddd4d1f-99c0-46b2-b072-4f2af3d9a76d,Marston,B-4,3.114335e+06,1.649458e+06,-105.094152,39.615852
2358,L&C LIQUORS,8100 W CRESTLINE AVE UNIT A-18,LIQUOR - STORE,LICENSE ISSUED - ACTIVE,10/31/2025 10:58:22 AM,12/14/2026 12:00:00 AM,264317.0,8100 W Crestline Ave Unit A18,Denver,CO,...,12010.0,3115317.0,1650209.0,8a09d97d-fbed-4d26-a721-a4200f27e2bc,Marston,B-4,3.115317e+06,1.650209e+06,-105.090655,39.617902
2370,MARTINI MARTINE LLC,8555 W BELLEVIEW AVE STE A03,LIQUOR - TASTINGS,LICENSE ISSUED - ACTIVE,6/16/2025 12:00:00 AM,6/16/2026 12:00:00 AM,270011.0,8555 W Belleview Ave Ste A03,Denver,CO,...,12014.0,3114424.0,1652660.0,5911c82c-d94b-450f-bdd7-9a39dd7fe1c1,Marston,B-2,3.114424e+06,1.652660e+06,-105.093786,39.624642
2380,BELLEVIEW WINE AND SPIRITS,7795 E BELLEVIEW 3,LIQUOR - STORE,LICENSE ISSUED - ACTIVE,2/24/2026 8:36:29 AM,2/12/2027 12:00:00 AM,32690148.0,7795 E Belleview Ave Unit 3,Denver,CO,...,6804.0,3169520.0,1652956.0,6710414d-cbc7-4a3a-9371-91cb04db16ef,Hampden South,B-8,3.169520e+06,1.652956e+06,-104.898188,39.624615
2386,VILLAGE WEST LIQUOR,8555 W BELLEVIEW AVE SUITE A03,LIQUOR - STORE,LICENSE ISSUED - ACTIVE,6/3/2025 10:24:42 AM,5/20/2026 12:00:00 AM,155994.0,8555 W Belleview Ave,Denver,CO,...,12014.0,3114526.0,1653012.0,59297ab8-88d5-4193-ac3d-078aeed9e791,Marston,B-2,3.114526e+06,1.653012e+06,-105.093418,39.625607


In [23]:
df_denver_liquor_stores_active_clean.to_csv('denver_liquor_stores.csv', index=False)

## Calculating walking distance
Using the Google Maps API to calculate walking distance between each Denver liquor store and my home address.
Then filtering for those where the walking distance is =<20 minutes -- deciding this is a reasonable "walking distance" for a 'portal' to Sonoma.

Docs for Google Maps Distance Matrix API: https://developers.google.com/maps/documentation/distance-matrix/start

### First - get a clean dataframe that has the addresses and coordinates of both the liquor stores in Denver and my home

In [ ]:
#Create a df with my address and all liquor stores

df_denver_liquor_stores_address = df_denver_liquor_stores_active_clean[['BUS_PROF_NAME','ADDRESS_LINE1','CITY','STATE','lon','lat']]
df_denver_liquor_stores_address.head()

,BUS_PROF_NAME,ADDRESS_LINE1,CITY,STATE,lon,lat
2353,COST PLUS WORLD MARKET,8601 W Cross Dr Unit CP,Denver,CO,-105.094152,39.615852
2358,L&C LIQUORS,8100 W Crestline Ave Unit A18,Denver,CO,-105.090655,39.617902
2370,MARTINI MARTINE LLC,8555 W Belleview Ave Ste A03,Denver,CO,-105.093786,39.624642
2380,BELLEVIEW WINE AND SPIRITS,7795 E Belleview Ave Unit 3,Denver,CO,-104.898188,39.624615
2386,VILLAGE WEST LIQUOR,8555 W Belleview Ave,Denver,CO,-105.093418,39.625607


In [ ]:
# #add my address
# my_address = {
#     'BUS_PROF_NAME': np.nan, 
#     'ADDRESS_LINE1': my_address,
#     'CITY': 'Denver',
#     'STATE': 'CO',
#     'lon': my_coords[1], 
#     'lat': my_coords[0]
#     }

In [ ]:
# df_my_address = pd.DataFrame([my_address])

In [ ]:
# dfs_to_concat = [df_denver_liquor_stores_address, df_my_address]
# df_all_address = pd.concat(dfs_to_concat)

In [25]:
df_all_address

NameError: name 'df_all_address' is not defined

In [ ]:
print(df_all_address.shape)
print(df_all_address['ADDRESS_LINE1'].notna().sum())
print(df_all_address['ADDRESS_LINE1'].head(10))

(222, 6)
206
2353          8601 W Cross Dr Unit CP
2358    8100 W Crestline Ave Unit A18
2370     8555 W Belleview Ave Ste A03
2380      7795 E Belleview Ave Unit 3
2386             8555 W Belleview Ave
2427             4827 S Wadsworth Way
2461                3706 W Quincy Ave
2464                8000 E Quincy Ave
2487        3533 S Monaco Street Pkwy
2492               8970 E Hampden Ave
Name: ADDRESS_LINE1, dtype: str


In [ ]:
duration_rows = []
addresses = []

for index, row in df_all_address.iterrows():
    address = row["ADDRESS_LINE1"]
    addresses.append(address)

    if pd.isna(row['ADDRESS_LINE1']):
        continue
    else:
        address_formatted = row['ADDRESS_LINE1'].replace(' ', '+').replace('#','')
        address_param = f"{address_formatted},+Denver,+CO"
        url = f"https://maps.googleapis.com/maps/api/distancematrix/json?destinations={address_param}&origins=39.728886%2C-104.9715073&mode=walking&key={api_key}"
    response = requests.get(url)
    data = response.json()

    for row_data in data.get("rows", []):
        for element in row_data.get("elements", []):
            duration_rows.append({
                "ADDRESS_LINE1": address,
                "status": element.get("status"),
                "duration_text": element.get("duration", {}).get("text"),
                "duration_value": element.get("duration", {}).get("value"),
            })

NameError: name 'df_all_address' is not defined

In [ ]:
df_duration = pd.DataFrame(duration_rows)

NameError: name 'duration_rows' is not defined

In [ ]:
df_duration.to_csv('duration_data.csv')

In [ ]:
df_duration.head()

In [ ]:
df_duration.dtypes

In [ ]:
df_all_address.dtypes

In [ ]:
from fuzzymatcher import fuzzy_left_join

df_all_address_durations = fuzzy_left_join(
    df_all_address,
    df_duration,
    left_on=["ADDRESS_LINE1"],
    right_on=["ADDRESS_LINE1"]
)

In [ ]:
df_all_address_durations.to_csv('denver_addresses_durations.csv')

In [ ]:
df_all_address_durations.head()

NameError: name 'df_all_address_durations' is not defined

'duration_value' represents duration in seconds. For this, I am looking for walks that are 15 minutes or less.
15 minutes = 900 seconds

In [ ]:
df_denver_liquor_stores_in_walking_distance = df_all_address_durations[df_all_address_durations['duration_value']<=900]

In [ ]:
df_denver_liquor_stores_in_walking_distance

In [ ]:
df_denver_liquor_stores_in_walking_distance.to_csv('denver_liquor_stores_walking_distance.csv')